# Publish SDK Skills to Unity Catalog

Publishes the six `skills/ontology-*` folders to the Unity Gateway skills library as `<skills_catalog>.<skills_schema>.<id>` (ids from `ci/skills.yaml`): create if missing, upload changed files under `/Skills/<catalog>/<schema>/<id>/`, finalize. Idempotent: a second run changes nothing. The schema must exist (this notebook reports, never creates it) and no grants are applied; see `docs/skills/README.md`.

In [0]:
# Wheel stable path (docs/distribution/wheel_publish.md): <sdk_volume_path>/.internal/genie_ontology_modeled_knowledge_sdk-<sdk_version>-py3-none-any.whl
# The skills schema is the SDK schema, so the wheel path is derived from the same widgets (passed as job parameters or set below).
dbutils.widgets.text("skills_catalog", "", "Skills catalog")
dbutils.widgets.text("skills_schema", "", "Skills schema (the SDK schema)")
dbutils.widgets.text("sdk_version", "0.1.1", "SDK wheel version")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "Dry run")
_cat, _sch = dbutils.widgets.get("skills_catalog").strip(), dbutils.widgets.get("skills_schema").strip()
if not _cat or not _sch:
    raise ValueError("Set skills_catalog and skills_schema (the SDK schema from `databricks bundle summary`)")
sdk_wheel = f"/Volumes/{_cat}/{_sch}/sdk_wheels/.internal/genie_ontology_modeled_knowledge_sdk-{dbutils.widgets.get('sdk_version').strip()}-py3-none-any.whl"
%pip install --upgrade databricks-sdk $sdk_wheel
dbutils.library.restartPython()

In [0]:
import json
from pathlib import Path

from databricks.sdk import WorkspaceClient
from genie_ontology_modeled_knowledge_sdk import skills_publish as sp

skills_catalog = dbutils.widgets.get("skills_catalog").strip()
skills_schema = dbutils.widgets.get("skills_schema").strip()
dry_run = dbutils.widgets.get("dry_run") == "true"

# SDK checkout = parent of notebooks/. Mapping: repo-root skills.yaml once moved by the owner, else the staged ci/skills.yaml.
sdk_root = Path.cwd().parent
skills_dir = sdk_root / "skills"
mapping_path = next(p for p in (sdk_root.parent / "skills.yaml", sdk_root / "ci" / "skills.yaml") if p.is_file())
mapping = sp.load_mapping(mapping_path, skills_catalog, skills_schema)

errors = [e for t in mapping for e in sp.skill_limit_errors(skills_dir / t.folder)]
if errors:
    raise ValueError("skill limit problems:\n" + "\n".join(errors))

w = WorkspaceClient()
report = sp.check_schema(w, skills_catalog, skills_schema)  # reports only; never creates
print(json.dumps(report, indent=2))
if not report["exists"]:
    raise RuntimeError(f"Skills schema missing or unreadable: {report['error']}")

plan = sp.plan_publish(skills_dir, mapping, w)
print(json.dumps({k: v.to_dict() for k, v in plan.items()}, indent=2))

In [0]:
changed = sp.publish(w, skills_dir, mapping, dry_run=dry_run)
result = {
    "dry_run": dry_run,
    "changes": sum(len(v) for v in changed.values()),
    "changed": changed,
    "published": sp.list_published(w, skills_catalog, skills_schema),
}
print(json.dumps(result, indent=2))
dbutils.notebook.exit(json.dumps(result))